In [1]:
df_train = pd.read_csv(train_csv)
df_train['filepath'] = df_train['image_name'].apply(
    lambda x: os.path.join(JPEG_DIR, 'train', f'{x}.jpg')
)

for path in df_train['filepath'].head(3):
    assert os.path.exists(path), f"Image not found: {path}"

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
df_train['fold'] = -1

for fold_idx, (_, val_idx) in enumerate(
    sgkf.split(df_train, df_train['target'], df_train['patient_id'])
):
    df_train.loc[val_idx, 'fold'] = fold_idx

print("Samples per fold:")
print(df_train['fold'].value_counts().sort_index())
print(f"\nMelanoma per fold:")
print(df_train[df_train['target'] == 1].groupby('fold').size())
print(f"\n✅ Folds created!")

NameError: name 'pd' is not defined

In [1]:
import nbformat
import glob

notebooks = glob.glob('experiments/exp5_deit_small.ipynb') + glob.glob('exp5_deit_small.ipynb')

for nb_path in notebooks:
    nb = nbformat.read(nb_path, as_version=4)
    
    changes = []
    
    # Check notebook-level
    if 'widgets' in nb.metadata:
        changes.append('notebook.metadata.widgets')
    
    # Check per-output
    for i, cell in enumerate(nb.cells):
        if 'outputs' in cell:
            for j, output in enumerate(cell.outputs):
                if 'metadata' in output and 'widgets' in output.metadata:
                    changes.append(f'cell[{i}].outputs[{j}].metadata.widgets')
    
    # Also report what's preserved (to reassure)
    n_text_outputs = sum(
        1 for cell in nb.cells if cell.get('outputs')
        for o in cell['outputs'] if o.get('output_type') == 'stream'
    )
    n_image_outputs = sum(
        1 for cell in nb.cells if cell.get('outputs')
        for o in cell['outputs'] if o.get('output_type') == 'display_data'
        and 'image/png' in o.get('data', {})
    )
    
    print(f'\n{nb_path}:')
    print(f'  Will remove: {len(changes)} widget metadata entries')
    print(f'  Will preserve: {n_text_outputs} text outputs, {n_image_outputs} images')


experiments/exp5_deit_small.ipynb:
  Will remove: 1 widget metadata entries
  Will preserve: 21 text outputs, 1 images


In [2]:
import nbformat
import os
import glob

# Clean all notebooks in the repo
notebooks = glob.glob('experiments/exp5_deit_small.ipynb') + glob.glob('exp5_deit_small.ipynb')

for nb_path in notebooks:
    nb = nbformat.read(nb_path, as_version=4)
    
    # Remove problematic widget metadata
    if 'widgets' in nb.metadata:
        del nb.metadata['widgets']
    
    # Also remove from each cell's outputs if present
    for cell in nb.cells:
        if 'outputs' in cell:
            for output in cell.outputs:
                if 'metadata' in output and 'widgets' in output.metadata:
                    del output.metadata['widgets']
    
    nbformat.write(nb, nb_path)
    print(f'✅ Cleaned: {nb_path}')

print('\nAll notebooks cleaned. Commit and push.')

✅ Cleaned: experiments/exp5_deit_small.ipynb

All notebooks cleaned. Commit and push.
